# TTA Validation on Gold-58 — Full Standalone
**VERSION: tta-validate-full-v1.0**

Validates whether test-time augmentation (8 views: identity, hflip, brightness ±0.10,
gamma 0.90/1.10, narrow/wide intensity window) improves macro AUC on the 58
expert-labeled gold studies, using the 0.89-LB EfficientNet-B3 checkpoint.

**Ship rule:** keep TTA iff full-TTA macro AUC − baseline macro AUC ≥ +0.003.

**Inputs (attach on Kaggle):**
- `effnet-best-ckpt` dataset (the 0.89 checkpoint, contains `effnet_best.pt`)
- `phase0runA`, `phase0runB`, `phase0runC` (Phase 0 image cache)
- Competition data (has `train_series/`)
- `gold_58.csv` (upload as a dataset, or place in /kaggle/input)

**Settings:** GPU ON (inference), Internet OFF. ~15 min for 58 studies.


In [ ]:
print("VERSION: tta-validate-full-v1.0")
import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from sklearn.metrics import roc_auc_score

print(f'torch {torch.__version__} | cuda: {torch.cuda.is_available()}')
assert torch.cuda.is_available(), 'FATAL: no GPU — enable GPU for inference'
DEVICE = torch.device('cuda:0')

# ---- constants (must match the 0.89 training run) ----
IMAGE_SIZE = 288
N_SLICES = 8
PLANES = ['axial', 'coronal', 'sagittal']
BACKBONE_NAME = 'efficientnet_b3'
PRETRAINED = False          # we load our own checkpoint, not ImageNet
DROP_PATH_RATE = 0.1
ATTN_DIM = 128
N_FINDINGS = 12
SEED = 42
LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA',
          'PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
assert len(LABELS) == 12

def find_under(root, name, max_depth=6):
    hits = []
    for dirpath, dirnames, filenames in os.walk(root):
        if dirpath.count(os.sep) - root.count(os.sep) > max_depth:
            dirnames[:] = []
            continue
        if name in filenames or name in dirnames:
            hits.append(os.path.join(dirpath, name))
    return hits

print('\nscanning /kaggle/input ...')
# ---- checkpoint ----
ckpt_hits = find_under('/kaggle/input', 'effnet_best.pt')
if not ckpt_hits:
    ckpt_hits = find_under('/kaggle/input', 'effnet_last.pt')
assert ckpt_hits, ('FATAL: no effnet_best.pt found under /kaggle/input. '
                   'Attach the effnet-best-ckpt dataset (0.89 checkpoint).')
CKPT_PATH = ckpt_hits[0]
print(f'checkpoint: {CKPT_PATH}')

# ---- gold csv ----
gold_hits = find_under('/kaggle/input', 'gold_58.csv')
assert gold_hits, 'FATAL: gold_58.csv not found — upload it as a Kaggle dataset'
GOLD_CSV = gold_hits[0]
print(f'gold csv: {GOLD_CSV}')

# ---- manifests (Phase 0 cache) ----
MANIFEST_PATHS = find_under('/kaggle/input', 'manifest.json')
assert MANIFEST_PATHS, 'FATAL: manifest.json not found — attach phase0runA/B/C'
print(f'manifests: {len(MANIFEST_PATHS)}')

# ---- train_series (gold studies are train studies) ----
COMP = None
for c in ['/kaggle/input/competitions/rsna-knee-abnormality-detection',
          '/kaggle/input/rsna-knee-abnormality-detection']:
    if os.path.isdir(c):
        COMP = c; break
assert COMP, 'FATAL: competition data not found'
print(f'competition: {COMP}')
print('setup: ALL GREEN')


In [ ]:
# ================= MODEL: EfficientNet-B3 2.5D + per-finding attention MIL ====
# Architecture identical to the 0.89 training run (v2.3 cell 7).
print('timm', timm.__version__)

class GatedAttentionMIL(nn.Module):
    def __init__(self, feat_dim, attn_dim=ATTN_DIM):
        super().__init__()
        self.V = nn.Linear(feat_dim, attn_dim)
        self.U = nn.Linear(feat_dim, attn_dim)
        self.w = nn.Linear(attn_dim, 1)
    def forward(self, h):  # h: (B, S, D)
        a = self.w(torch.tanh(self.V(h)) * torch.sigmoid(self.U(h)))
        a = F.softmax(a, dim=1)
        return (a * h).sum(dim=1), a.squeeze(-1)

class KneeEffNetMIL(nn.Module):
    def __init__(self, backbone_name=BACKBONE_NAME, attn_dim=ATTN_DIM,
                 n_findings=N_FINDINGS, pretrained=PRETRAINED,
                 drop_path_rate=DROP_PATH_RATE):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                                          num_classes=0, drop_path_rate=drop_path_rate)
        D = self.backbone.num_features
        print(f'backbone feature dim: {D}')
        self.mil_heads = nn.ModuleList([GatedAttentionMIL(D, attn_dim)
                                        for _ in range(n_findings)])
        self.heads = nn.ModuleList([nn.Linear(D, 1) for _ in range(n_findings)])
    def forward_plane(self, x):  # x: (B, S, 3, H, W)
        B, S = x.shape[:2]
        h = self.backbone(x.reshape(B * S, *x.shape[2:])).reshape(B, S, -1)
        return torch.stack([mil(h)[0] for mil in self.mil_heads], dim=1)
    def forward(self, planes, plane_mask):
        feats = torch.stack([self.forward_plane(planes[p]) for p in PLANES], dim=1)
        m = plane_mask.reshape(-1, 3, 1, 1)
        fused = (feats * m).sum(1) / m.sum(1).clamp(min=1e-6)
        return torch.cat([self.heads[f](fused[:, f, :])
                          for f in range(N_FINDINGS)], dim=1)

# ---- load checkpoint (prefer EMA weights = the 0.89 submission weights) ----
model = KneeEffNetMIL().to(DEVICE)
ck = torch.load(CKPT_PATH, map_location='cpu')
print(f"checkpoint keys: {sorted(ck.keys())} | epoch={ck.get('epoch')} best={ck.get('best')}")
if ck.get('ema') is not None:
    model.load_state_dict(ck['ema'], strict=True)
    print('loaded EMA weights (the 0.89 submission weights)')
else:
    model.load_state_dict(ck['model'], strict=True)
    print('loaded model weights (no EMA in checkpoint)')
model.eval()
# shape probe
with torch.no_grad():
    _pb = {p: torch.randn(1, 4, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE) for p in PLANES}
    _pm = torch.ones(1, 3).to(DEVICE)
    _o = model(_pb, _pm)
    print(f'shape probe: -> {tuple(_o.shape)}')
    assert _o.shape == (1, N_FINDINGS)
print('model: ALL GREEN')


In [ ]:
# ================= DATA: Phase 0 cache index + inference plane loader ====
class ShardStore:
    def __init__(self, manifest_paths):
        self.studies = {}
        self._z = {}
        for mp in manifest_paths:
            cache_dir = str(Path(mp).parent)
            man = json.load(open(mp))
            for uid, entry in man['studies'].items():
                if uid in self.studies:
                    raise RuntimeError(f'duplicate study {uid} across caches')
                self.studies[uid] = (cache_dir, entry)
        print(f'indexed {len(self.studies)} studies from {len(manifest_paths)} manifests')
    def get_plane(self, uid, plane):
        pid = os.getpid()
        cache_dir, entry = self.studies[uid]
        shard = entry['shard']
        key = (pid, cache_dir, shard)
        z = self._z.get(key)
        if z is None:
            z = np.load(os.path.join(cache_dir, shard))
            self._z[key] = z
            while len(self._z) > 4:
                k = next(iter(self._z))
                try: self._z.pop(k).close()
                except Exception: pass
        return z[f'{uid}/{plane}']
    def planes_of(self, uid):
        return list(self.studies[uid][1]['planes'].keys())

store = ShardStore(MANIFEST_PATHS)

def sample_slices(vol, n):
    S = vol.shape[0]
    if S == n: return vol
    idx = np.linspace(0, S - 1, n).round().astype(int)
    return vol[idx]

def load_study_planes(uid):
    """Load one study -> ({plane: (S,3,H,W) float32 [0,1]}, mask (3,)).
    No augmentation. Missing planes zero-filled (mask 0)."""
    planes, mask = {}, []
    have = set(store.planes_of(uid))
    for p in PLANES:
        if p in have:
            vol = store.get_plane(uid, p)                      # (S,336,336) uint8
            vol = sample_slices(vol, N_SLICES).astype(np.float32) / 255.0
            t = torch.from_numpy(vol).unsqueeze(1)              # (S,1,H,W)
            t = F.interpolate(t, size=(IMAGE_SIZE, IMAGE_SIZE),
                              mode='bilinear', align_corners=False)
            vol = t.squeeze(1).numpy().astype(np.float32)       # (S,288,288)
            planes[p] = np.repeat(vol[:, None, :, :], 3, axis=1).astype(np.float32)
            mask.append(1.0)
        else:
            planes[p] = np.zeros((N_SLICES, 3, IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)
            mask.append(0.0)
    return planes, np.array(mask, dtype=np.float32)

# ---- gold labels ----
gold = pd.read_csv(GOLD_CSV)
_gcols = {c.lower().strip(): c for c in gold.columns}
uid_col = _gcols.get('studyinstanceuid') or _gcols.get('study_instance_uid')
assert uid_col, 'FATAL: no StudyInstanceUID column in gold_58.csv'
y_cols = []
for lab in LABELS:
    key = lab.lower().strip()
    assert key in _gcols, f'FATAL: no gold column for {lab!r}'
    y_cols.append(_gcols[key])
print(f'mapped {len(y_cols)} gold label columns OK')
gold_uids = gold[uid_col].astype(str).tolist()
Y_TRUE = gold[y_cols].to_numpy(dtype=np.float64)
assert Y_TRUE.shape == (len(gold_uids), 12)
print(f'{len(gold_uids)} gold studies | positives per finding: {(Y_TRUE > 0.5).sum(0).tolist()}')
# check cache coverage
missing = [u for u in gold_uids if u not in store.studies]
print(f'gold studies missing from cache: {len(missing)}')
if missing: print('  e.g.', missing[:3])
print('data: ALL GREEN')


In [ ]:
# ================= TTA: 8 views, intensity-only + hflip =================
# Mirrors tta_inference.py. Transforms run in [0,1]; the 0.89 checkpoint is
# ImageNet-EfficientNet-B3 so no [-1,1] remap is needed.
def _v_identity(v): return v
def _v_hflip(v):    return v[:, :, :, ::-1].copy()   # L/R mirror-safe
def _v_bup(v):      return np.clip(v + 0.10, 0, 1).astype(np.float32)
def _v_bdn(v):      return np.clip(v - 0.10, 0, 1).astype(np.float32)
def _v_g090(v):     return np.clip(np.power(v, 0.90), 0, 1).astype(np.float32)
def _v_g110(v):     return np.clip(np.power(v, 1.10), 0, 1).astype(np.float32)
def _v_wnar(v):     return np.clip((v - 0.15) / 0.70, 0, 1).astype(np.float32)
def _v_wsoft(v):    return (v * 0.90 + 0.05).astype(np.float32)

TTA_VIEWS = [('identity', _v_identity), ('hflip', _v_hflip),
             ('bright_up', _v_bup), ('bright_down', _v_bdn),
             ('gamma_090', _v_g090), ('gamma_110', _v_g110),
             ('window_narrow', _v_wnar), ('window_soft', _v_wsoft)]
assert len(TTA_VIEWS) == 8
print(f'{len(TTA_VIEWS)} TTA views: {[n for n, _ in TTA_VIEWS]}')

@torch.no_grad()
def predict_study(uid, planes, pmask, views):
    """Run one study through `views` (list of (name, fn)). Returns (12,) probs."""
    model.eval()
    per_plane = {p: [] for p in PLANES}
    for _name, fn in views:
        for pi, p in enumerate(PLANES):
            vol = planes[p]
            if pmask[pi] == 1.0:
                vol = fn(vol)          # transform only present planes
            per_plane[p].append(vol)  # missing planes stay zero-filled
    batch = {p: torch.from_numpy(np.stack(per_plane[p], 0)).to(DEVICE) for p in PLANES}
    mask_t = torch.from_numpy(np.tile(pmask[None, :], (len(views), 1))).to(DEVICE)
    logits = model(batch, mask_t)
    probs = torch.sigmoid(logits).double().mean(dim=0).cpu().numpy()
    assert probs.shape == (N_FINDINGS,) and np.all(np.isfinite(probs))
    return probs

# quick transform sanity check
_v = np.random.rand(2, 3, 32, 32).astype(np.float32)
for _n, _f in TTA_VIEWS:
    _o = _f(_v)
    assert _o.shape == _v.shape and _o.dtype == np.float32
    assert _o.min() >= 0 and _o.max() <= 1, _n
print('TTA transforms: ALL GREEN')


In [ ]:
# ================= INFERENCE: baseline vs full TTA on gold-58 =================
@torch.no_grad()
def run_views(views, tag):
    P = np.zeros((len(gold_uids), N_FINDINGS))
    ok = np.ones(len(gold_uids), dtype=bool)
    t0 = time.time()
    for i, uid in enumerate(gold_uids):
        try:
            if uid not in store.studies:
                raise KeyError('uid not in cache')
            planes, pmask = load_study_planes(uid)
            P[i] = predict_study(uid, planes, pmask, views)
        except Exception as e:
            ok[i] = False
            P[i] = 0.5
            if (not ok).sum() <= 5:
                print(f'  WARN {uid}: {type(e).__name__}: {e}')
        if (i + 1) % 10 == 0 or (i + 1) == len(gold_uids):
            print(f'  {tag}: {i+1}/{len(gold_uids)} ({time.time()-t0:.0f}s)', flush=True)
    return P, ok

def macro_auc(P, Y, ok):
    aucs = []
    for f in range(N_FINDINGS):
        yt = Y[ok, f]
        if yt.min() == yt.max() or (yt > 0.5).sum() < 2 or (yt <= 0.5).sum() < 2:
            aucs.append(np.nan); continue
        try: aucs.append(roc_auc_score(yt, P[ok, f]))
        except Exception: aucs.append(np.nan)
    return float(np.nanmean(aucs)), aucs

IDENTITY = [TTA_VIEWS[0]]
print('\n[1/3] baseline (identity only)...')
P_base, ok = run_views(IDENTITY, 'baseline')
macro_base, aucs_base = macro_auc(P_base, Y_TRUE, ok)
print(f'baseline macro AUC: {macro_base:.4f} (n_ok={int(ok.sum())}/{len(gold_uids)})')

print('\n[2/3] full TTA (8 views)...')
P_tta, _ = run_views(TTA_VIEWS, 'tta8')
macro_tta, aucs_tta = macro_auc(P_tta, Y_TRUE, ok)
print(f'full-TTA macro AUC: {macro_tta:.4f}  (delta {macro_tta - macro_base:+.4f})')


In [ ]:
# ================= VERDICT =================
print('\n[3/3] per-view ablation (identity + one view)...')
print(f"  {'view':14s} {'macro':>7s} {'delta':>8s}  verdict")
harmful = []
for name, fn in TTA_VIEWS[1:]:
    P_v, _ = run_views([TTA_VIEWS[0], (name, fn)], f'abl_{name}')
    m_v, _ = macro_auc(P_v, Y_TRUE, ok)
    d = m_v - macro_base
    if d < -0.001:
        harmful.append(name); verdict = 'DROP (harmful)'
    elif d >= 0: verdict = 'keep'
    else: verdict = 'neutral'
    print(f'  {name:14s} {m_v:7.4f} {d:+8.4f}  {verdict}')

delta = macro_tta - macro_base
print('\n===================== VERDICT =====================')
print(f'  baseline macro AUC: {macro_base:.4f}')
print(f'  full TTA macro AUC: {macro_tta:.4f}   (delta {delta:+.4f})')
if harmful:
    print(f'  HARMFUL views (drop before shipping): {harmful}')
if delta >= 0.003:
    print('  => SHIP TTA' + (f' (minus {harmful})' if harmful else ''))
else:
    print('  => DO NOT SHIP TTA (delta < +0.003)')
print('===================================================\n')
print('per-finding AUC (baseline -> full TTA):')
for lab, a0, a1 in zip(LABELS, aucs_base, aucs_tta):
    print(f'  {lab:16s} {a0:.4f} -> {a1:.4f}  ({a1 - a0:+.4f})')
